In [ ]:
import ee
import geemap.core as geemap
from datetime import datetime

ee.Authenticate()
ee.Initialize(project='sat-model-1')
print(ee.String('Hello from the Earth Engine servers!').getInfo())

In [ ]:
import pandas as pd

In [ ]:
df = pd.read_csv('model.csv', sep=';')

start_date = '2025-02-02'
end_date = '2025-02-04'

etopo1 = ee.Image("NOAA/NGDC/ETOPO1").select('bedrock')
        
oisst = ee.ImageCollection("NOAA/CDR/OISST/V2_1") \
    .filterDate(start_date, end_date) \
    .mean() \
    .select('sst')

In [ ]:
def get_sst_at_point(image, lon, lat):
    point = ee.Geometry.Point([lon, lat])
    value = image.sample(point, 5000).first().get('sst')
    return value.getInfo() * 0.01  # масштабный коэффициент

df['sst'] = [get_sst_at_point(oisst, lon, lat) for lon, lat in zip(df['lon'], df['lat'])]

In [ ]:
def get_depth_at_point(image, lon, lat):
    point = ee.Geometry.Point([float(lon), float(lat)])
    
    # ETOPO1 resolution is 1 arc-minute (~1855m at equator)
    stats = image.reduceRegion(
        reducer=ee.Reducer.first(),
        geometry=point.buffer(1000),
        scale=1855,
        maxPixels=1e9
    )
    
    try:
        value = stats.get('bedrock').getInfo()
        # ETOPO1: negative = ocean depth, positive = land elevation
        return value
    except Exception as _:
        return None
        
    return None

df['depth'] = [get_depth_at_point(etopo1, lon, lat) for lon, lat in zip(df['lon'], df['lat'])]

In [ ]:
df.head(10)

In [ ]:
df.to_csv('result.csv')

In [ ]:

m = geemap.Map()

features = []
for idx, row in df.iterrows():
    feat = ee.Feature(
        ee.Geometry.Point([row['lon'], row['lat']]),
        {
            'id': idx,
            'depth': row.get('depth', None)  # Add your depth data as property
        }
    )
    features.append(feat)

fc = ee.FeatureCollection(features)

# Style based on depth (optional visualization)
vis_params = {
    'color': 'red',
    'pointSize': 8,
    'pointShape': 'circle'
}

m.addLayer(fc, vis_params, 'Points with Depth')

In [ ]:

vis_params = {
        'min': -6000,
        'max': 0,
        'palette': ['lightblue', 'blue', 'darkblue', 'black'][::-1]
    }
    
m.add_layer(etopo1, vis_params, 'Глубина (Copernicus)', opacity=0.6)

m.setCenter(150, 56, 2) 

In [ ]:
sst_vis = {
    'min': -5,
    'max': 5,
    'palette': [
        '440154',  # Фиолетовый (холодная)
        '31688e',
        '35b779',
        'fde725'   # Жёлтый (тёплая)
    ]
}

m.add_layer(oisst, sst_vis, 'Темература', opacity=0.6)

In [ ]:
m